# XGBoost Model Training — Delta Exchange India

**Self-contained** notebook: no local `agent` package needed.  
Trains three XGBoost classifiers (15m / 1h / 4h) using live candle data from the Delta Exchange India public API.

## Quick-start
1. Run **Cell 1** (install deps — only needed once per Colab session)
2. Run **Cell 2** (config — set your symbol / Google Drive path)
3. Run **Cell 3** (all imports)
4. Run **Cell 4** (Delta Exchange data fetcher)
5. Run **Cell 5** (feature engineering)
6. Run **Cell 6** (trainer class)
7. Run **Cell 7** (execute training)

In [ ]:
# ── Cell 1 · Install dependencies ─────────────────────────────────────────────
# Run once per Colab session.  Runtime → Run all will handle the restart.
import importlib, subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

missing = []
for pkg, import_name in [
    ('xgboost',  'xgboost'),
    ('ta',       'ta'),
    ('pandas',   'pandas'),
    ('numpy',    'numpy'),
    ('requests', 'requests'),
    ('scikit-learn', 'sklearn'),
]:
    try:
        importlib.import_module(import_name)
    except ModuleNotFoundError:
        missing.append(pkg)

if missing:
    print(f'Installing: {missing}')
    _pip(*missing)
    print('Done — you may need to restart the runtime once (Runtime → Restart runtime), then re-run from Cell 2.')
else:
    print('All dependencies already installed ✓')

In [ ]:
# ── Cell 2 · Configuration ─────────────────────────────────────────────────────
from pathlib import Path

# ── User-configurable ──────────────────────────────────────────────────────────
SYMBOL          = 'BTCUSD'        # Delta Exchange perpetual symbol
TIMEFRAMES      = ['15m', '1h', '4h']
CANDLES_LIMIT   = 3000            # candles per timeframe

# ── Delta Exchange India API credentials ───────────────────────────────────────
# The historical candles endpoint is PUBLIC and does not require authentication.
# This notebook always uses unauthenticated requests for /v2/history/candles.
#
# API keys are OPTIONAL:
#   • You may still configure them here if you plan to extend the
#     notebook to call private/account endpoints in the future.
#   • For this training workflow, keys are not needed.
#
# HOW TO SET YOUR KEYS (choose one method):
#
# Method 1 — Colab Secrets (recommended, safest):
#   Left sidebar → 🔑 Secrets → add keys named
#   'DELTA_API_KEY' and 'DELTA_API_SECRET'  (toggle 'Notebook access' ON)
#   Then leave the two lines below as-is (empty strings).
#
# Method 2 — Paste directly (quick, but avoid sharing the notebook):
#   Replace the empty strings below with your actual key and secret.

DELTA_API_KEY    = ''   # e.g. 'abcdef1234567890'
DELTA_API_SECRET = ''   # e.g. 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# Auto-load from Colab Secrets if the variables above are left empty
try:
    from google.colab import userdata
    if not DELTA_API_KEY:
        DELTA_API_KEY    = userdata.get('DELTA_API_KEY')    or ''
    if not DELTA_API_SECRET:
        DELTA_API_SECRET = userdata.get('DELTA_API_SECRET') or ''
except Exception:
    pass  # not in Colab, or secrets not configured — will run unauthenticated

if DELTA_API_KEY and DELTA_API_SECRET:
    print('API credentials loaded ✓ (authenticated mode — for private endpoints only)')
else:
    print('No API credentials — running in unauthenticated mode (public candles endpoint only)')

# ── Save location ──────────────────────────────────────────────────────────────
# Option A — Google Drive (survives session restart, recommended):
#   SAVE_DIR = Path('/content/drive/MyDrive/models')
# Option B — local /content (lost when session ends):
SAVE_DIR = Path('/content/models')

# ── Label thresholds (% move in the *next* candle) ─────────────────────────────
BUY_THRESHOLD  =  0.5   # return > +0.5 %  → BUY  (label 2)
SELL_THRESHOLD = -0.5   # return < -0.5 %  → SELL (label 0)
#                                           → HOLD (label 1)

# ── XGBoost hyper-parameters ───────────────────────────────────────────────────
XGB_PARAMS = dict(
    max_depth            = 6,
    learning_rate        = 0.05,
    n_estimators         = 300,
    subsample            = 0.8,
    colsample_bytree     = 0.8,
    min_child_weight     = 5,
    objective            = 'multi:softprob',
    num_class            = 3,
    random_state         = 42,
    eval_metric          = 'mlogloss',
    early_stopping_rounds= 20,
)

# Mount Google Drive if SAVE_DIR is under /content/drive
if str(SAVE_DIR).startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f'Config OK — symbol={SYMBOL}, timeframes={TIMEFRAMES}, save_dir={SAVE_DIR}')

In [ ]:
# ── Cell 3 · Imports ───────────────────────────────────────────────────────────
import time
import pickle
import logging
import warnings
from datetime import datetime, timezone, timedelta
from typing import Dict, List, Optional, Tuple

import numpy  as np
import pandas as pd
import requests
import ta
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings('ignore')
logging.basicConfig(
    level  = logging.INFO,
    format = '%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt= '%H:%M:%S',
)
log = logging.getLogger('trainer')
print('Imports OK ✓')

In [ ]:
# ── Cell 4 · Delta Exchange India — data fetcher ───────────────────────────────
# Historical candles endpoint is PUBLIC — API keys are optional.
# When keys ARE provided (Cell 2), every request is HMAC-signed for
# authenticated access (higher rate limits, future account endpoints).

import hashlib
import hmac

BASE_URL = 'https://api.india.delta.exchange'

_RESOLUTION_MAP = {
    '1m' : '1m',  '5m' : '5m',  '15m': '15m', '30m': '30m',
    '1h' : '1h',  '2h' : '2h',  '4h' : '4h',  '1d' : '1d',
}

_INTERVAL_SECONDS = {
    '1m' : 60,    '5m' : 300,   '15m': 900,   '30m': 1800,
    '1h' : 3600,  '2h' : 7200,  '4h' : 14400, '1d' : 86400,
}


def _build_auth_headers(
    method: str,
    path: str,
    query_string: str,
    body: str = '',
    api_key: str = '',
    api_secret: str = '',
) -> Dict[str, str]:
    """Build Delta Exchange HMAC-SHA256 authentication headers.

    Signature scheme (from Delta Exchange docs):
        signature = HMAC-SHA256(api_secret,
                        method + timestamp + path + '?' + query_string + body)

    Returns an empty dict if no credentials are configured.
    """
    if not api_key or not api_secret:
        return {}

    timestamp = str(int(time.time()))  # Unix seconds as string
    qs        = f'?{query_string}' if query_string else ''
    message   = method.upper() + timestamp + path + qs + body

    signature = hmac.new(
        api_secret.encode('utf-8'),
        message.encode('utf-8'),
        hashlib.sha256,
    ).hexdigest()

    return {
        'api-key'  : api_key,
        'timestamp': timestamp,
        'signature': signature,
    }


def _fetch_candles_chunk(
    symbol: str,
    resolution: str,
    start: int,
    end: int,
    session: requests.Session,
    retries: int = 3,
) -> List[Dict]:
    """Fetch one chunk of candles, signed if credentials are available."""
    path   = '/v2/history/candles'
    url    = BASE_URL + path
    params = {
        'resolution': _RESOLUTION_MAP[resolution],
        'symbol'    : symbol,
        'start'     : start,
        'end'       : end,
    }
    query_string = '&'.join(f'{k}={v}' for k, v in sorted(params.items()))

    # For public candles endpoint — do NOT send auth headers.
    # Delta Exchange India documentation states historical candles are public
    # and do not require authentication. Using auth headers here can cause 401s
    # if the signature format or key permissions do not match expectations.
    auth_headers: Dict[str, str] = {}

    for attempt in range(1, retries + 1):
        try:
            resp = session.get(url, params=params, headers=auth_headers, timeout=20)
            resp.raise_for_status()
            data = resp.json()
            if data.get('success') is False:
                raise ValueError(f'API error: {data}')

            # Handle both possible response shapes:
            # 1) {"result": {"candles": [...]}}
            # 2) {"result": [...]} (result is the candles array directly)
            result = data.get('result')
            if result is None:
                raise ValueError(f'API error: no result in response: {data}')

            if isinstance(result, dict):
                candles = result.get('candles', [])
            elif isinstance(result, list):
                candles = result
            else:
                raise ValueError(f'Unexpected result type in response: {type(result)}')

            return candles
        except Exception as exc:
            log.warning(f'Attempt {attempt}/{retries} failed: {exc}')
            if attempt < retries:
                time.sleep(2 ** attempt)
    raise RuntimeError(f'All {retries} attempts failed for {symbol} {resolution}')


def fetch_historical_data(
    symbol: str,
    resolution: str,
    limit: int = 3000,
) -> pd.DataFrame:
    """Fetch `limit` candles and return a clean OHLCV DataFrame.

    The Delta Exchange API returns a maximum of ~500 candles per request so
    this function pages backwards automatically until `limit` candles are
    collected or no earlier data exists.
    """
    if resolution not in _INTERVAL_SECONDS:
        raise ValueError(f'Unsupported resolution {resolution!r}. Choose from {list(_INTERVAL_SECONDS)}')

    step = _INTERVAL_SECONDS[resolution]
    now  = int(datetime.now(timezone.utc).timestamp())

    # Delta allows max 2000 candles in one call — we page in 500-candle chunks
    CHUNK = 500
    all_candles: List[Dict] = []

    end_ts = now
    session = requests.Session()
    session.headers.update({'Accept': 'application/json'})

    log.info(f'Fetching {limit} x {resolution} candles for {symbol}…')

    while len(all_candles) < limit:
        fetch_n  = min(CHUNK, limit - len(all_candles))
        start_ts = end_ts - fetch_n * step

        chunk = _fetch_candles_chunk(symbol, resolution, start_ts, end_ts, session)
        if not chunk:
            log.info('No more historical data available from API.')
            break

        all_candles = chunk + all_candles   # prepend older candles
        end_ts = start_ts - step            # move window back
        time.sleep(0.2)                     # be polite to the API

    if not all_candles:
        raise RuntimeError(f'Zero candles returned for {symbol} {resolution}')

    # ── Normalise to DataFrame ────────────────────────────────────────────────
    df = pd.DataFrame(all_candles)

    # Delta returns 'time' as Unix seconds
    df.rename(columns={'time': 'timestamp'}, inplace=True)

    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['datetime'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df.sort_values('timestamp', inplace=True)
    df.drop_duplicates('timestamp', inplace=True)
    df.reset_index(drop=True, inplace=True)

    # Sanity-check required columns
    missing = [c for c in ['open', 'high', 'low', 'close', 'volume'] if c not in df.columns]
    if missing:
        raise ValueError(f'Missing OHLCV columns: {missing}.  Got: {df.columns.tolist()}')

    log.info(f'Fetched {len(df)} candles — {df["datetime"].iloc[0]} → {df["datetime"].iloc[-1]}')
    return df


print('Data fetcher defined ✓')

In [ ]:
# ── Cell 5 · Feature engineering ──────────────────────────────────────────────
# All features are computed in a single vectorised pass over the full DataFrame.
# This is orders of magnitude faster than the original O(n²) per-row approach.

def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """Compute technical features in-place and return the enriched DataFrame."""
    c = df['close'].copy()
    h = df['high'].copy()
    l = df['low'].copy()
    v = df['volume'].copy()

    # ── Momentum ──────────────────────────────────────────────────────────────
    df['rsi_14']  = ta.momentum.RSIIndicator(c, window=14).rsi()
    df['rsi_7']   = ta.momentum.RSIIndicator(c, window=7).rsi()
    stoch = ta.momentum.StochasticOscillator(h, l, c, window=14, smooth_window=3)
    df['stoch_k'] = stoch.stoch()
    df['stoch_d'] = stoch.stoch_signal()
    df['roc_5']   = ta.momentum.ROCIndicator(c, window=5).roc()
    df['roc_10']  = ta.momentum.ROCIndicator(c, window=10).roc()
    df['cci_14']  = ta.trend.CCIIndicator(h, l, c, window=14).cci()
    df['williams_r'] = ta.momentum.WilliamsRIndicator(h, l, c, lbp=14).williams_r()

    # ── Trend ─────────────────────────────────────────────────────────────────
    for span in [9, 21, 50, 100]:
        df[f'ema_{span}'] = ta.trend.EMAIndicator(c, window=span).ema_indicator()
    for span in [20, 50]:
        df[f'sma_{span}'] = ta.trend.SMAIndicator(c, window=span).sma_indicator()

    macd = ta.trend.MACD(c, window_slow=26, window_fast=12, window_sign=9)
    df['macd']        = macd.macd()
    df['macd_signal'] = macd.macd_signal()
    df['macd_hist']   = macd.macd_diff()

    df['adx_14'] = ta.trend.ADXIndicator(h, l, c, window=14).adx()
    df['dmp_14'] = ta.trend.ADXIndicator(h, l, c, window=14).adx_pos()
    df['dmn_14'] = ta.trend.ADXIndicator(h, l, c, window=14).adx_neg()

    # ── Volatility ────────────────────────────────────────────────────────────
    boll = ta.volatility.BollingerBands(c, window=20, window_dev=2)
    df['bb_upper'] = boll.bollinger_hband()
    df['bb_lower'] = boll.bollinger_lband()
    df['bb_mid']   = boll.bollinger_mavg()
    df['bb_width'] = boll.bollinger_wband()
    df['bb_pct']   = boll.bollinger_pband()

    df['atr_14']   = ta.volatility.AverageTrueRange(h, l, c, window=14).average_true_range()
    df['atr_7']    = ta.volatility.AverageTrueRange(h, l, c, window=7).average_true_range()

    # ── Volume ────────────────────────────────────────────────────────────────
    df['obv']          = ta.volume.OnBalanceVolumeIndicator(c, v).on_balance_volume()
    df['vol_ema_20']   = ta.trend.EMAIndicator(v, window=20).ema_indicator()
    df['vol_ratio']    = v / (df['vol_ema_20'].replace(0, np.nan))

    # ── Price-derived ratios ──────────────────────────────────────────────────
    df['close_ema9_ratio']  = c / df['ema_9'].replace(0, np.nan)
    df['close_ema21_ratio'] = c / df['ema_21'].replace(0, np.nan)
    df['close_ema50_ratio'] = c / df['ema_50'].replace(0, np.nan)
    df['ema9_ema21_ratio']  = df['ema_9'] / df['ema_21'].replace(0, np.nan)
    df['ema21_ema50_ratio'] = df['ema_21'] / df['ema_50'].replace(0, np.nan)
    df['high_low_ratio']    = h / l.replace(0, np.nan)
    df['close_open_ratio']  = c / df['open'].replace(0, np.nan)

    # ── Returns ───────────────────────────────────────────────────────────────
    df['ret_1']  = c.pct_change(1)
    df['ret_3']  = c.pct_change(3)
    df['ret_5']  = c.pct_change(5)
    df['ret_10'] = c.pct_change(10)

    # Normalised ATR (ATR / close) — scale-free volatility measure
    df['natr_14'] = df['atr_14'] / c.replace(0, np.nan)

    return df


def create_labels(
    df: pd.DataFrame,
    forward_periods: int = 1,
    buy_threshold: float = BUY_THRESHOLD,
    sell_threshold: float = SELL_THRESHOLD,
) -> np.ndarray:
    """Return-based labels: 0=SELL, 1=HOLD, 2=BUY."""
    future_close  = df['close'].shift(-forward_periods)
    return_pct    = (future_close - df['close']) / df['close'] * 100

    labels = np.ones(len(df), dtype=int)          # default HOLD
    labels[return_pct >  buy_threshold]  = 2      # BUY
    labels[return_pct <  sell_threshold] = 0      # SELL
    labels[-forward_periods:]            = 1      # last rows: no future → HOLD
    return labels


# Feature columns (all float columns added by add_features, excluding OHLCV / timestamp)
_EXCLUDED = {'timestamp', 'datetime', 'open', 'high', 'low', 'close', 'volume'}

def get_feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c not in _EXCLUDED]


print('Feature engineering defined ✓')

In [ ]:
# ── Cell 6 · Trainer ──────────────────────────────────────────────────────────

def train_model(
    X: pd.DataFrame,
    y: np.ndarray,
    timeframe: str,
    xgb_params: dict,
) -> Tuple[XGBClassifier, Dict]:
    """Train + evaluate XGBoost on 70/15/15 temporal split."""
    n = len(X)
    t1 = int(n * 0.70)
    t2 = int(n * 0.85)

    X_tr, y_tr = X.iloc[:t1],     y[:t1]
    X_vl, y_vl = X.iloc[t1:t2],   y[t1:t2]
    X_te, y_te = X.iloc[t2:],     y[t2:]

    if len(X_vl) == 0 or len(X_te) == 0:
        raise ValueError(f'Not enough samples for train/val/test split (n={n})')

    # Class-balanced sample weights (handles HOLD dominance)
    sw_tr = compute_sample_weight('balanced', y_tr)

    # Separate early_stopping_rounds from fit-time params vs constructor params
    fit_params = {k: v for k, v in xgb_params.items() if k not in ('early_stopping_rounds',)}
    early_stop = xgb_params.get('early_stopping_rounds', 20)

    model = XGBClassifier(**fit_params, early_stopping_rounds=early_stop)

    t0 = time.time()
    model.fit(
        X_tr, y_tr,
        sample_weight = sw_tr,
        eval_set      = [(X_vl, y_vl)],
        verbose       = 50,
    )
    elapsed = time.time() - t0

    metrics = {
        'train_accuracy': float(model.score(X_tr, y_tr)),
        'val_accuracy'  : float(model.score(X_vl, y_vl)),
        'test_accuracy' : float(model.score(X_te, y_te)),
        'training_time' : round(elapsed, 2),
        'n_estimators_used': model.best_iteration + 1 if hasattr(model, 'best_iteration') and model.best_iteration else xgb_params['n_estimators'],
    }

    log.info(
        f'[{timeframe}] train={metrics["train_accuracy"]:.4f}  '
        f'val={metrics["val_accuracy"]:.4f}  '
        f'test={metrics["test_accuracy"]:.4f}  '
        f'time={metrics["training_time"]}s'
    )

    # Detailed classification report on test set
    y_pred = model.predict(X_te)
    print(f'\n── {timeframe} test-set report ──')
    print(classification_report(y_te, y_pred, target_names=['SELL', 'HOLD', 'BUY'], zero_division=0))

    return model, metrics


def save_model(
    model: XGBClassifier,
    symbol: str,
    timeframe: str,
    feature_cols: List[str],
    metrics: Dict,
    save_dir: Path,
) -> Path:
    """Pickle the model + metadata bundle and verify the saved file."""
    fname    = f'xgboost_classifier_{symbol}_{timeframe}.pkl'
    out_path = save_dir / fname

    # Back up existing file
    if out_path.exists():
        bak = out_path.with_suffix('.pkl.bak')
        out_path.rename(bak)
        log.info(f'Existing model backed up to {bak}')

    bundle = {
        'model'       : model,
        'feature_cols': feature_cols,
        'symbol'      : symbol,
        'timeframe'   : timeframe,
        'metrics'     : metrics,
        'trained_at'  : datetime.now(timezone.utc).isoformat(),
    }

    with open(out_path, 'wb') as fh:
        pickle.dump(bundle, fh, protocol=pickle.HIGHEST_PROTOCOL)

    # ── Verify ────────────────────────────────────────────────────────────────
    with open(out_path, 'rb') as fh:
        loaded = pickle.load(fh)

    assert isinstance(loaded['model'], XGBClassifier), 'Saved object is not XGBClassifier!'
    assert hasattr(loaded['model'], 'predict_proba'),  'Model missing predict_proba!'

    dummy = np.random.rand(1, len(feature_cols))
    loaded['model'].predict(dummy)
    loaded['model'].predict_proba(dummy)

    log.info(f'Model saved & verified → {out_path}  ({out_path.stat().st_size:,} bytes)')
    return out_path


print('Trainer defined ✓')

In [ ]:
# ── Cell 7 · Run training ──────────────────────────────────────────────────────

print('=' * 64)
print(f'XGBoost Training  |  symbol={SYMBOL}  |  timeframes={TIMEFRAMES}')
print('=' * 64)

all_results = []

for tf in TIMEFRAMES:
    print(f'\n{"─" * 50}')
    print(f'  Timeframe: {tf}')
    print(f'{"─" * 50}')

    # 1. Fetch data ────────────────────────────────────────────────────────────
    try:
        df = fetch_historical_data(SYMBOL, tf, limit=CANDLES_LIMIT)
    except Exception as exc:
        print(f'✗  Data fetch failed for {tf}: {exc}')
        continue

    if len(df) < 500:
        print(f'✗  Only {len(df)} candles returned — need ≥500. Skipping.')
        continue

    # 2. Feature engineering ───────────────────────────────────────────────────
    df = add_features(df)
    feature_cols = get_feature_cols(df)
    print(f'   Features computed: {len(feature_cols)}')

    # 3. Labels ────────────────────────────────────────────────────────────────
    df['label'] = create_labels(df)
    label_counts = pd.Series(df['label']).value_counts().sort_index()
    print(f'   Labels  →  SELL={label_counts.get(0,0)}  HOLD={label_counts.get(1,0)}  BUY={label_counts.get(2,0)}')

    # 4. Drop rows with NaN features (warm-up rows from indicators)
    clean = df[feature_cols + ['label']].dropna()
    if len(clean) < 200:
        print(f'✗  Only {len(clean)} clean rows after dropna — skipping.')
        continue

    X = clean[feature_cols]
    y = clean['label'].values
    print(f'   Clean samples: {len(X)}')

    # 5. Train ─────────────────────────────────────────────────────────────────
    try:
        model, metrics = train_model(X, y, tf, XGB_PARAMS)
    except Exception as exc:
        print(f'✗  Training failed for {tf}: {exc}')
        import traceback; traceback.print_exc()
        continue

    # 6. Save ──────────────────────────────────────────────────────────────────
    try:
        saved_path = save_model(model, SYMBOL, tf, feature_cols, metrics, SAVE_DIR)
        print(f'✓  Model saved → {saved_path}')
    except Exception as exc:
        print(f'✗  Save failed: {exc}')
        continue

    all_results.append({'timeframe': tf, 'samples': len(X), 'features': len(feature_cols), **metrics, 'path': str(saved_path)})

# ── Summary ───────────────────────────────────────────────────────────────────
print('\n' + '=' * 64)
print('Training complete!')
print('=' * 64)

if all_results:
    summary_df = pd.DataFrame(all_results)
    summary_path = SAVE_DIR / 'training_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(f'Summary saved → {summary_path}')
    display(summary_df[['timeframe', 'samples', 'features', 'train_accuracy', 'val_accuracy', 'test_accuracy', 'training_time']])
else:
    print('No models were successfully trained.')

In [ ]:
# ── Cell 8 · (Optional) Load & inspect a saved model ──────────────────────────
import pickle
from pathlib import Path

INSPECT_TF = '1h'   # change to '15m' or '4h' as needed

model_path = SAVE_DIR / f'xgboost_classifier_{SYMBOL}_{INSPECT_TF}.pkl'

if not model_path.exists():
    print(f'Model not found: {model_path}')
    print('Run Cell 7 first and ensure at least one timeframe trained successfully.')
    # Create a placeholder bundle so the summary prints below do not fail.
    bundle = {
        'symbol': SYMBOL,
        'timeframe': INSPECT_TF,
        'trained_at': 'N/A',
        'feature_cols': [],
        'metrics': {},
        'model': None,
    }
else:
    with open(model_path, 'rb') as fh:
        bundle = pickle.load(fh)

print(f'Loaded: {model_path}')
print(f'Symbol     : {bundle["symbol"]}')
print(f'Timeframe  : {bundle["timeframe"]}')
print(f'Trained at : {bundle["trained_at"]}')
print(f'Features   : {len(bundle["feature_cols"])}')
print(f'Metrics    : {bundle["metrics"]}')
print(f'Model type : {type(bundle["model"]).__name__}')